**Introduction**
Random forests are popular statistical learning algorithms. Some of their primary benefits include reducing variance, bias, and the chance of overfitting.

This activity is a continuation of the project We began modeling with decision trees for an airline. Here, will train, tune, and evaluate a random forest model using data from spreadsheet of survey responses from 129,880 customers. It includes data points such as class, flight distance, and inflight entertainment. Your random forest model will be used to predict whether a customer will be satisfied with their flight experience.


In [1]:
# Import the relevant functions from `sklearn.ensemble`, `sklearn.model_selection`, and `sklearn.metrics`.
 
import numpy as np
import pandas as pd

import pickle as pkl
 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, PredefinedSplit, GridSearchCV
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

In [3]:
#Import the data
air_data = pd.read_csv('Invistico_Airline.csv')
air_data.head()

,satisfaction,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,...,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,satisfied,Female,Loyal Customer,65.0,Personal Travel,Eco,265.0,0,0.0,0,...,2,3,3,0,3,5,3,2,0,0.0
1,satisfied,Male,Loyal Customer,47.0,Personal Travel,Business,NaN,0,0.0,0,...,2,3,4,4,4,2,3,2,310,305.0
2,satisfied,Female,Loyal Customer,15.0,Personal Travel,Eco,2138.0,0,0.0,0,...,2,2,3,3,4,4,4,2,0,0.0
3,satisfied,Female,Loyal Customer,60.0,Personal Travel,Eco,623.0,0,0.0,0,...,3,1,1,0,1,4,1,3,0,0.0
4,satisfied,Female,Loyal Customer,99.0,Personal Travel,NaN,354.0,0,0.0,0,...,4,2,2,0,2,4,2,5,0,0.0


**Data cleaning

In [4]:
# Display variable names and types.
air_data.dtypes

satisfaction                          object
Gender                                object
Customer Type                         object
Age                                  float64
Type of Travel                        object
Class                                 object
Flight Distance                      float64
Seat comfort                           int64
Departure/Arrival time convenient    float64
Food and drink                         int64
Gate location                        float64
Inflight wifi service                float64
Inflight entertainment                 int64
Online support                         int64
Ease of Online booking                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Online boarding                        int64
Departure Delay in Minutes             int64
Arrival De

**Question: What do you observe about the differences in data types among the variables included in the data?**

There are three types of variables included in the data: int64, float64, and object. The object variables are satisfaction, customer type, gender, type of travel, and class.

In [5]:
# Identify the number of rows and the number of columns.
air_data.shape

(129880, 23)

In [7]:
#Check for missing values
air_data.isna().any(axis=1).sum()

418

**Question: How many rows of data are missing values?****

There are 418 rows with missing values.

In [8]:
# Drop missing values.
air_data_subset = air_data.dropna(axis=0)

In [10]:
# Count of missing values.
air_data_subset.isna().any(axis=1).sum()

0

In [11]:
# Convert categorical features to one-hot encoded features.
air_data_subset = pd.get_dummies(air_data_subset,
                                columns = ['Customer Type','Type of Travel','Class'])

In [12]:
air_data_subset.head()

,satisfaction,Gender,Age,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,...,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes,Customer Type_Loyal Customer,Customer Type_disloyal Customer,Type of Travel_Business travel,Type of Travel_Personal Travel,Class_Business,Class_Eco,Class_Eco Plus
0,satisfied,Female,65.0,265.0,0,0.0,0,2.0,2.0,4,...,2,0,0.0,1,0,0,1,0,1,0
2,satisfied,Female,15.0,2138.0,0,0.0,0,3.0,2.0,0,...,2,0,0.0,1,0,0,1,0,1,0
3,satisfied,Female,60.0,623.0,0,0.0,0,3.0,3.0,4,...,3,0,0.0,1,0,0,1,0,1,0
5,satisfied,Male,30.0,1894.0,0,0.0,0,3.0,2.0,0,...,2,0,0.0,1,0,0,1,0,1,0
6,satisfied,Female,66.0,227.0,0,0.0,0,3.0,2.0,5,...,3,17,15.0,1,0,0,1,0,1,0


In [15]:
air_data_subset = air_data_subset.drop('Gender', axis=1)
air_data_subset.head()

,satisfaction,Age,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,...,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes,Customer Type_Loyal Customer,Customer Type_disloyal Customer,Type of Travel_Business travel,Type of Travel_Personal Travel,Class_Business,Class_Eco,Class_Eco Plus
0,satisfied,65.0,265.0,0,0.0,0,2.0,2.0,4,2,...,2,0,0.0,1,0,0,1,0,1,0
2,satisfied,15.0,2138.0,0,0.0,0,3.0,2.0,0,2,...,2,0,0.0,1,0,0,1,0,1,0
3,satisfied,60.0,623.0,0,0.0,0,3.0,3.0,4,3,...,3,0,0.0,1,0,0,1,0,1,0
5,satisfied,30.0,1894.0,0,0.0,0,3.0,2.0,0,2,...,2,0,0.0,1,0,0,1,0,1,0
6,satisfied,66.0,227.0,0,0.0,0,3.0,2.0,5,5,...,3,17,15.0,1,0,0,1,0,1,0


In [16]:
# Display variables.
air_data_subset.dtypes

satisfaction                          object
Age                                  float64
Flight Distance                      float64
Seat comfort                           int64
Departure/Arrival time convenient    float64
Food and drink                         int64
Gate location                        float64
Inflight wifi service                float64
Inflight entertainment                 int64
Online support                         int64
Ease of Online booking                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Cleanliness                            int64
Online boarding                        int64
Departure Delay in Minutes             int64
Arrival Delay in Minutes             float64
Customer Type_Loyal Customer           uint8
Customer Type_disloyal Customer        uint8
Type of Travel_Business travel         uint8
Type of Tr

**Question: What changes do you observe after converting the string data to dummy variables?**

All of the following changes could be observed:

Customer Type --> Customer Type_Loyal Customer and Customer Type_disloyal Customer
Type of Travel --> Type of Travel_Business travel and Type of Travel_Personal travel
Class --> Class_Business, Class_Eco, Class_Eco Plus

**Model building**
The first step to building your model is separating the labels (y) from the features (X).

In [18]:
# Separate the dataset into labels (y) and features (X).
y = air_data_subset['satisfaction']

X = air_data_subset.drop('satisfaction', axis=1)

In [19]:
# Separate into train, validate, test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25,
                                                   random_state = 0)

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size = 0.25, random_state = 0)

**Tune the model**
Now, fit and tune a random forest model with separate validation set. Begin by determining a set of hyperparameters for tuning the model using GridSearchCV.

In [20]:
# Determine set of hyperparameters.

cv_params = {'n_estimators' : [50,100], 
              'max_depth' : [10,50],        
              'min_samples_leaf' : [0.5,1], 
              'min_samples_split' : [0.001, 0.01],
              'max_features' : ["sqrt"], 
              'max_samples' : [.5,.9]}

In [23]:
# Create list of split indices.
split_index = [0 if x in X_val.index else -1 for x in X_train.index]
custom_split = PredefinedSplit(split_index)

Use list comprehension, iterating over the indices of X_train. The list can consists of 0s to indicate data points that should be treated as validation data and -1s to indicate data points that should be treated as training data.

Use PredfinedSplit(), passing in split_index, saving the output as custom_split. This will serve as a custom split that will identify which data points from the train set should be treated as validation data during GridSearch.

In [21]:
# Instantiate model.
rf = RandomForestClassifier(random_state = 0)

In [24]:
# Search over specified parameters.
rf_val = GridSearchCV(rf, cv_params, cv = custom_split, refit='f1', n_jobs = -1, verbose = 1)

In [25]:
%%time

# Fit the model.
rf_val.fit(X_train, y_train)

Fitting 1 folds for each of 32 candidates, totalling 32 fits
CPU times: total: 14.2 s
Wall time: 42.4 s


GridSearchCV(cv=PredefinedSplit(test_fold=array([-1, -1, ...,  0,  0])),
             estimator=RandomForestClassifier(random_state=0), n_jobs=-1,
             param_grid={'max_depth': [10, 50], 'max_features': ['sqrt'],
                         'max_samples': [0.5, 0.9],
                         'min_samples_leaf': [0.5, 1],
                         'min_samples_split': [0.001, 0.01],
                         'n_estimators': [50, 100]},
             refit='f1', verbose=1)

In [26]:
#Obtain the best params
rf_val.best_params_

{'max_depth': 50,
 'max_features': 'sqrt',
 'max_samples': 0.9,
 'min_samples_leaf': 1,
 'min_samples_split': 0.001,
 'n_estimators': 100}

**Results and evaluation**
Use the selected model to predict on your test data. Use the optimal parameters found via GridSearchCV.

In [27]:
# Use optimal parameters on GridSearchCV.
rf_opt = RandomForestClassifier(n_estimators = 100, max_depth = 50, 
                                min_samples_leaf = 1, min_samples_split = 0.001,
                                max_features="sqrt", max_samples = 0.9, random_state = 0)

In [28]:
# Fit the optimal model.
rf_opt.fit(X_train, y_train)

RandomForestClassifier(max_depth=50, max_samples=0.9, min_samples_split=0.001,
                       random_state=0)

In [29]:
# Predict on test set.
y_pred = rf_opt.predict(X_test)

**Obtain performance scores**
First, get  precision score.

In [43]:
# Get precision score.
pc_test = precision_score(y_test, y_pred, pos_label = "satisfied")
print("The precision score is: " '%0.3f' % pc_test)

# Get recall score.
rc_test = recall_score(y_test, y_pred, pos_label = "satisfied")
print("The Recall score is: " '%0.3f' % rc_test)


The precision score is: 0.952
The Recall score is: 0.949


In [44]:
# Get F1 score.
f1_test = f1_score(y_test, y_pred, pos_label = "satisfied")
print("The F1 score is: " '%0.3f' % f1_test)

# Get accuracy score.
ac_test = accuracy_score(y_test, y_pred)
print("The Accuracy score is: " '%0.3f' % ac_test)

The F1 score is: 0.951
The Accuracy score is: 0.946


**Pros:**

1. The coding workload is reduced.
2. The scripts for data splitting are shorter.
3. It's only necessary to evaluate test dataset performance once, instead of two evaluations (validate and test).
**Cons:**

1. If a model is evaluated using samples that were also used to build or fine-tune that model, it likely will provide a biased evaluation.
2. A potential overfitting issue could happen when fitting the model's scores on the test data.

**Evaluate the model**

**Question: What are the four basic parameters for evaluating the performance of a classification model?**

True positives (TP): These are correctly predicted positive values, which means the value of actual and predicted classes are positive.

True negatives (TN): These are correctly predicted negative values, which means the value of the actual and predicted classes are negative.

False positives (FP): This occurs when the value of the actual class is negative and the value of the predicted class is positive.

False negatives (FN): This occurs when the value of the actual class is positive and the value of the predicted class in negative.

**Question: What do the four scores demonstrate about your model, and how do you calculate them?**

Accuracy (TP+TN/TP+FP+FN+TN): The ratio of correctly predicted observations to total observations.

Precision (TP/TP+FP): The ratio of correctly predicted positive observations to total predicted positive observations.

Recall (Sensitivity, TP/TP+FN): The ratio of correctly predicted positive observations to all observations in actual class.

F1 score: The harmonic average of precision and recall, which takes into account both false positives and false negatives.

Calculate the scores: precision score, recall score, accuracy score, F1 score.

In [48]:
# Precision score on test data set.
print("\nThe precision score is: {pc:.3f}".format(pc = pc_test), "for the test set,", "\nwhich means of all positive predictions,", "\n{pc_pct:.1f}% prediction are true positive.".format(pc_pct = pc_test * 100))


The precision score is: 0.952 for the test set, 
which means of all positive predictions, 
95.2% prediction are true positive.


In [47]:
# Recall score on test data set.

print("\nThe recall score is: {rc:.3f}".format(rc = rc_test), "for the test set,", "\nwhich means of which means of all real positive cases in test set,", "\n{rc_pct:.1f}% are  predicted positive.".format(rc_pct = rc_test * 100))


The recall score is: 0.949 for the test set, 
which means of which means of all real positive cases in test set, 
94.9% are  predicted positive.


In [49]:
# Accuracy score on test data set.
print("\nThe accuracy score is: {ac:.3f}".format(ac = ac_test), "for the test set,", "\nwhich means of all cases in test set,", "{ac_pct:.1f}% are predicted true positive or true negative.".format(ac_pct = ac_test * 100))


The accuracy score is: 0.946 for the test set, 
which means of all cases in test set, 94.6% are predicted true positive or true negative.


In [50]:
# F1 score on test data set.
print("\nThe F1 score is: {f1:.3f}".format(f1 = f1_test), "for the test set,", "\nwhich means the test set's harmonic mean is {f1_pct:.1f}%.".format(f1_pct = f1_test * 100))


The F1 score is: 0.951 for the test set, 
which means the test set's harmonic mean is 95.1%.


**Question: How does this model perform based on the four scores?**

The model performs well according to all 4 performance metrics. The model's precision score is slightly better than the 3 other metrics.

Finally, create a table of results that you can use to evaluate the performace of your model.

In [51]:
# Create table of results.
table = pd.DataFrame({'Model' : ['Tuned Decision Tree', 'Tuned Random Forest'],
                      'F1': [0.947552, f1_test],
                      'Recall': [0.940103, rc_test],
                      'Precision': [0.955123, pc_test],
                      'Accuracy': [0.943077, ac_test]
             })

table

,Model,F1,Recall,Precision,Accuracy
0,Tuned Decision Tree,0.947552,0.940103,0.955123,0.943077
1,Tuned Random Forest,0.950825,0.949406,0.952249,0.946147


**Question: How does the random forest model compare to the decision tree model you built in the previous lab?**

The tuned random forest has higher scores overall, so it is the better model. Particularly, it shows a better F1 score than the decision tree model, which indicates that the random forest model may do better at classification when taking into account false positives and false negatives.

**Considerations**
**What are the key takeaways from this lab?**

- Data exploring, cleaning, and encoding are necessary for model building.
- A separate validation set is typically used for tuning a model, rather than using the test set. This also helps avoid the evaluation becoming biased.
- F1 scores are usually more useful than accuracy scores. If the cost of false positives and false negatives are very different, it’s better to use the F1 score and combine the information from precision and recall.
- The random forest model yields a more effective performance than a decision tree model.
**What summary would you provide to stakeholders?**

The random forest model predicted satisfaction with more than 94.61% accuracy. The precision is over 95% and the recall is approximately 94.9%.
The random forest model outperformed the tuned decision tree with the best hyperparameters in most of the four scores. This indicates that the random forest model may perform better.

Because stakeholders were interested in learning about the factors that are most important to customer satisfaction, this would be shared based on the tuned random forest.
